# 7 — Batches and cross-entropy loss

**Before:** notebook **6** (full GPT).

**This notebook:** `get_batch` + one training-style forward on real token IDs.

**Learning objectives**

- Sample random contiguous batches from tokenized Shakespeare.
- Move data to CPU or GPU and run one training-style forward.
- Read a single batch loss value.
- Explain why batching is required for efficient training.


In [ ]:
# --- Setup: find repo root (llm-c-from-scratch or Cursor workbook) ---
import sys
from pathlib import Path


def find_llm_root() -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "llmc" / "__init__.py").is_file():
            return base
        nested = base / "llm-c-from-scratch"
        if (nested / "llmc" / "__init__.py").is_file():
            return nested
    return Path.cwd()


ROOT = find_llm_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from llmc.notebook_utils import data_path, checkpoint_path

DATA = data_path(ROOT)
CHECKPOINT = checkpoint_path(ROOT)
print("ROOT", ROOT.resolve())
print("data", "OK" if DATA.is_file() else "missing")


In [ ]:
import torch
from llmc.data import CharTokenizer, get_batch, load_text, train_val_split
from llmc.model import GPT, GPTConfig

text = load_text(DATA)
train_text, val_text = train_val_split(text)
tok = CharTokenizer.from_text(text)
train_ids = torch.tensor(tok.encode(train_text), dtype=torch.long)
val_ids = torch.tensor(tok.encode(val_text), dtype=torch.long)

config = GPTConfig.tiny(vocab_size=tok.vocab_size, block_size=64)
model = GPT(config)


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
x, y = get_batch(train_ids, batch_size=8, block_size=config.block_size, device=device)
_, loss = model(x, y)
print(f"device={device}, batch loss={loss.item():.4f}")
